# Project 30 — LongevityAI: Biological Age & Healthy-Aging Intelligence

## Real NHANES biomarkers → model-derived biomarker age → healthy-aging research signals

**Portfolio objective:** build a rigorous, reproducible longevity-data-science pipeline that estimates a model-derived biomarker age from routine physiological and laboratory measurements, then audits age-acceleration patterns, robustness, subgroup error and the biomarkers carrying the most age signal.

> **Scientific framing:** this notebook does not claim to measure a clinically validated “true biological age,” diagnose disease, predict lifespan, or prove that changing any biomarker will slow aging. The output is a research-oriented age proxy learned from cross-sectional NHANES data.

### Skills demonstrated
- real health-data ingestion and validation
- leakage-aware feature policy
- regression baselines and non-linear ensembles
- cross-validation and locked test evaluation
- bias-corrected biomarker age acceleration
- permutation importance
- subgroup error and robustness auditing
- research-safe longevity interpretation


## 1. Imports, reproducibility and data provenance

The analysis uses a public merged NHANES biomarker table derived from the US CDC National Health and Nutrition Examination Survey. The source table is hosted in the public epenning/aging-biomarkers research repository.

Primary provenance:
- CDC NHANES: https://www.cdc.gov/nchs/nhanes/
- merged public analysis table: https://github.com/epenning/aging-biomarkers
- biological-age methodology context: Kwon & Belsky, GeroScience (2021), BioAge toolkit


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)

DATA_URL = "https://raw.githubusercontent.com/epenning/aging-biomarkers/main/nhanes_data.csv"
print("Reproducibility seed:", SEED)
print("Dataset URL:", DATA_URL)


In [ ]:
df_raw = pd.read_csv(DATA_URL)
print("Raw shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())


## 2. Data-quality audit

We inspect types, missingness, duplicate participant IDs and plausible age coverage before modelling. Export-only row-number fields are removed because they carry no biological information.


In [ ]:
df = df_raw.copy()
unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
if unnamed:
    df = df.drop(columns=unnamed)

quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

print("Shape after housekeeping:", df.shape)
print("Duplicate participant IDs:", int(df["ID"].duplicated().sum()))
print("Age range:", (df["Age"].min(), df["Age"].max()))
display(quality)


## 3. Adult cohort and leakage-safe feature policy

The model is restricted to adults aged 18+. Participant ID is excluded. Chronological age is the target and is never used as a predictor. Gender is retained as a demographic covariate.


In [ ]:
adult = df.loc[df["Age"] >= 18].copy()
adult["Gender"] = adult["Gender"].map({"Female": 0, "Male": 1}).astype(float)

TARGET = "Age"
FEATURES = [c for c in adult.columns if c not in ["ID", TARGET]]
X = adult[FEATURES].copy()
y = adult[TARGET].astype(float).copy()

print("Adult cohort:", len(adult))
print("Predictor count:", len(FEATURES))
print("Predictors:", FEATURES)
assert TARGET not in FEATURES
assert "ID" not in FEATURES


## 4. Exploratory aging signal


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(y, bins=20, edgecolor="black", alpha=0.8)
ax.set_title("Chronological age distribution — adult NHANES cohort")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Participants")
plt.tight_layout()
plt.show()

corr = adult[FEATURES + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=np.abs, ascending=False)
print("Correlation with chronological age:")
display(corr.to_frame("correlation_with_age"))


In [ ]:
num = adult[FEATURES + [TARGET]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(num.values, aspect="auto", vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(num.columns)))
ax.set_xticklabels(num.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(num.index)))
ax.set_yticklabels(num.index, fontsize=8)
ax.set_title("Biomarker correlation matrix")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 5. Locked train/test split

A fixed 80/20 split creates a test set untouched during model selection. Broad age bins are used for stratification so the train and test sets preserve age-range coverage.


In [ ]:
age_bins = pd.cut(y, bins=[17,29,39,49,59,69,120], labels=False, include_lowest=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=age_bins
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train age:", (y_train.min(), y_train.max()), "| Test age:", (y_test.min(), y_test.max()))


## 6. Baseline and candidate models

We compare a mean-age baseline, regularised linear regression and three non-linear ensemble approaches. Missing values are imputed with training-fold medians inside each pipeline.


In [ ]:
models = {
    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),
    "GradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=250, learning_rate=0.03, max_depth=2,
            min_samples_leaf=12, loss="huber", random_state=SEED
        ))
    ]),
    "RandomForest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=350, max_depth=10, min_samples_leaf=5,
            max_features=0.8, n_jobs=-1, random_state=SEED
        ))
    ]),
    "HistGradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
            l2_regularization=1.0, random_state=SEED
        ))
    ])
}

baseline_pred = np.repeat(y_train.mean(), len(y_test))
print("Mean-age baseline test MAE:", round(mean_absolute_error(y_test, baseline_pred), 3))


## 7. Five-fold cross-validation model comparison


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1
    )
    rows.append({
        "model": name,
        "cv_mae_mean": -scores["test_mae"].mean(),
        "cv_mae_std": scores["test_mae"].std(),
        "cv_rmse_mean": -scores["test_rmse"].mean(),
        "cv_r2_mean": scores["test_r2"].mean()
    })

cv_results = pd.DataFrame(rows).sort_values("cv_mae_mean").reset_index(drop=True)
display(cv_results.round(4))
BEST_NAME = cv_results.loc[0, "model"]
print("Selected by CV MAE:", BEST_NAME)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(cv_results["model"], cv_results["cv_mae_mean"], yerr=cv_results["cv_mae_std"], capsize=4)
ax.set_ylabel("5-fold CV MAE (years; lower is better)")
ax.set_title("LongevityAI model comparison")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## 8. Locked test evaluation


In [ ]:
best_model = clone(models[BEST_NAME]).fit(X_train, y_train)
pred = best_model.predict(X_test)

TEST_MAE = mean_absolute_error(y_test, pred)
TEST_RMSE = mean_squared_error(y_test, pred) ** 0.5
TEST_R2 = r2_score(y_test, pred)

print(f"Best model: {BEST_NAME}")
print(f"Test MAE: {TEST_MAE:.3f} years")
print(f"Test RMSE: {TEST_RMSE:.3f} years")
print(f"Test R²: {TEST_R2:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(y_test, pred, alpha=0.45, s=24)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, "--", linewidth=2, label="Perfect agreement")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Chronological age")
ax.set_ylabel("Predicted biomarker age proxy")
ax.set_title("Model-derived biomarker age vs chronological age")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Correcting regression-to-the-mean bias

Raw age-prediction residuals are usually biased by chronological age. We estimate the residual-age trend on the training set only and remove that expected bias from the test predictions before defining age acceleration.


In [ ]:
train_pred = best_model.predict(X_train)
train_gap = train_pred - y_train.to_numpy()

bias_fit = np.polyfit(y_train.to_numpy(), train_gap, deg=1)
expected_bias_test = np.polyval(bias_fit, y_test.to_numpy())
corrected_pred = pred - expected_bias_test
age_acceleration = corrected_pred - y_test.to_numpy()

print("Training residual-age bias slope:", round(float(bias_fit[0]), 4))
print("Corrected age-acceleration mean:", round(float(age_acceleration.mean()), 4))
print("Corrected age-acceleration SD:", round(float(age_acceleration.std(ddof=1)), 4))
print("Correlation(corrected gap, chronological age):", round(float(np.corrcoef(age_acceleration, y_test)[0,1]), 4))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(age_acceleration, bins=25, edgecolor="black", alpha=0.8)
ax.axvline(0, linestyle="--", linewidth=2)
ax.set_xlabel("Bias-corrected biomarker age acceleration (years)")
ax.set_ylabel("Participants")
ax.set_title("Research age-acceleration distribution")
plt.tight_layout()
plt.show()


## 10. Biomarker importance

Permutation importance measures predictive age signal, not causal importance and not intervention value.


In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="neg_mean_absolute_error", n_repeats=20,
    random_state=SEED, n_jobs=-1
)
importance = pd.DataFrame({
    "feature": FEATURES,
    "mae_increase": perm.importances_mean,
    "std": perm.importances_std
}).sort_values("mae_increase", ascending=False)
display(importance.round(4))


In [ ]:
top = importance.head(10).sort_values("mae_increase")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(top["feature"], top["mae_increase"], xerr=top["std"])
ax.set_xlabel("Increase in MAE after permutation (years)")
ax.set_title("Top biomarker age signals — permutation importance")
plt.tight_layout()
plt.show()


## 11. Subgroup error audit

A longevity model that performs well overall may still behave unevenly across demographic or age groups. We therefore report error by sex and age band.


In [ ]:
audit = X_test.copy()
audit["chronological_age"] = y_test.values
audit["predicted_age"] = pred
audit["abs_error"] = np.abs(pred - y_test.values)
audit["age_acceleration"] = age_acceleration
audit["sex"] = np.where(audit["Gender"] == 1, "Male", "Female")
audit["age_band"] = pd.cut(
    audit["chronological_age"], [17,29,39,49,59,69,120],
    labels=["18-29","30-39","40-49","50-59","60-69","70+"]
)

sex_audit = audit.groupby("sex").agg(
    n=("abs_error","size"),
    MAE=("abs_error","mean"),
    mean_age_gap=("age_acceleration","mean")
)
age_audit = audit.groupby("age_band", observed=False).agg(
    n=("abs_error","size"),
    MAE=("abs_error","mean"),
    mean_age_gap=("age_acceleration","mean")
)

print("Sex audit")
display(sex_audit.round(3))
print("Age-band audit")
display(age_audit.round(3))


## 12. Longevity research segmentation

For exploratory portfolio purposes only, the corrected age gap is divided into:
- younger biomarker profile: at or below -5 years
- age-consistent profile: between -5 and +5 years
- older biomarker profile: at or above +5 years

These are not clinical categories.


In [ ]:
def segment_gap(g):
    if g <= -5:
        return "Younger biomarker profile"
    if g >= 5:
        return "Older biomarker profile"
    return "Age-consistent profile"

audit["research_segment"] = [segment_gap(x) for x in audit["age_acceleration"]]
segment_summary = audit.groupby("research_segment").agg(
    n=("age_acceleration","size"),
    mean_gap=("age_acceleration","mean"),
    median_gap=("age_acceleration","median"),
    mean_chronological_age=("chronological_age","mean")
).sort_values("mean_gap")
display(segment_summary.round(2))


## 13. Robustness — model agreement

All candidate models are refit on the same training sample. Prediction correlations reveal whether the inferred aging signal is broadly shared across model families or highly model-dependent.


In [ ]:
pred_matrix = {}
for name, model in models.items():
    fitted = clone(model).fit(X_train, y_train)
    pred_matrix[name] = fitted.predict(X_test)

pred_df = pd.DataFrame(pred_matrix)
agreement = pred_df.corr()
print("Prediction correlation across model families:")
display(agreement.round(3))


## 14. Anti-aging and longevity relevance

This project demonstrates a complete biomarker-clock engineering workflow:
1. validate a real biomarker cohort;
2. learn normative age signal from physiology;
3. quantify out-of-sample performance;
4. correct residual age bias before interpreting age gaps;
5. expose predictive biomarker signals;
6. audit subgroup reliability;
7. separate predictive association from causal anti-aging claims.

A real longevity R&D team could extend this framework with longitudinal biomarkers, epigenetic clocks, proteomics, metabolomics, wearable signals, intervention trials and hard outcomes such as morbidity or mortality.


In [ ]:
TOP_FEATURES = importance.head(5)["feature"].tolist()
MAX_SEX_MAE_GAP = float(sex_audit["MAE"].max() - sex_audit["MAE"].min())

print("PROJECT 30 — LONGEVITYAI FINAL EXECUTIVE SUMMARY")
print("=" * 78)
print(f"Real adult NHANES participants analysed: {len(adult):,}")
print(f"Biomarker / demographic predictors: {len(FEATURES)}")
print(f"Selected model: {BEST_NAME}")
print(f"Locked-test MAE: {TEST_MAE:.2f} years")
print(f"Locked-test RMSE: {TEST_RMSE:.2f} years")
print(f"Locked-test R²: {TEST_R2:.3f}")
print(f"Bias-corrected age-gap mean: {age_acceleration.mean():.2f} years")
print(f"Gap-age correlation after correction: {np.corrcoef(age_acceleration, y_test)[0,1]:.3f}")
print("Top predictive age signals:", ", ".join(TOP_FEATURES))
print(f"Sex subgroup MAE spread: {MAX_SEX_MAE_GAP:.2f} years")
print()
print("SOLUTION")
print("Use this pipeline as a research-grade biomarker-age intelligence layer for")
print("longevity studies, model benchmarking and cohort stratification — not as a")
print("clinical diagnosis, lifespan forecast, or proof that modifying a feature reverses aging.")
print()
print("PROJECT 30 COMPLETE ✅")


In [ ]:
# Integrity checks
assert len(adult) > 3000
assert len(FEATURES) >= 10
assert np.isfinite(TEST_MAE) and TEST_MAE < 20
assert np.isfinite(TEST_R2)
assert abs(np.mean(age_acceleration)) < 3
assert set(importance["feature"]) == set(FEATURES)
assert audit["research_segment"].notna().all()
print("All Project 30 integrity checks passed ✅")
